# Whitespace Locations - Silver Layer

Generate expansion candidate locations from top H3 cells by population AND POI count with overlap reduction.

**Filtering Strategy:**
1. Filter to top 25% H3 cells by BOTH population AND total POI count (configurable via `percentile_threshold`)
2. Exclude candidates within minimum distance of existing stores (configurable via `min_distance_to_store_miles`)
3. Apply greedy spacing to avoid oversaturation (configurable via `min_candidate_spacing_miles`)
4. Spatial join with Census county subdivisions to get city/town names

**Inputs:**
- `{catalog}.{silver_schema}.h3_features_clean` - Clean H3 features (filtered to expansion state)
- `{catalog}.{bronze_schema}.current_stores_ne` - Current store locations for distance calculation

**Output:**
- `{catalog}.{silver_schema}.whitespace_locations` - Expansion candidate locations

**Output Schema:**
- location_id, store_type, latitude, longitude
- address, city (spatial join), zip_code, state, country_code, geo_accuracy
- distance_to_nearest_current_store (Haversine miles)
- nearest_store_id (ID of nearest existing store)
- h3_cell_id, total_poi_count, population, urbanity

## Parameters

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, lit, row_number, udf, broadcast, collect_list, struct
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, ArrayType, StringType
import math

# Notebook parameters
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("bronze_schema", "")
dbutils.widgets.text("silver_schema", "")
dbutils.widgets.text("state_filter", "MA")
dbutils.widgets.text("location_id_start", "999001")
dbutils.widgets.text("min_distance_to_store_miles", "1.0")
dbutils.widgets.text("min_candidate_spacing_miles", "0.5")  # 0.5 miles between candidates
dbutils.widgets.text("percentile_threshold", "0.75")  # Top 25%

# Extract parameters
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
state_filter = dbutils.widgets.get("state_filter")
location_id_start = int(dbutils.widgets.get("location_id_start"))
min_distance_to_store_miles = float(dbutils.widgets.get("min_distance_to_store_miles"))
min_candidate_spacing_miles = float(dbutils.widgets.get("min_candidate_spacing_miles"))
percentile_threshold = float(dbutils.widgets.get("percentile_threshold"))

assert catalog and bronze_schema and silver_schema, "Missing required parameters"

# Table names
h3_features_table = f"{catalog}.{silver_schema}.h3_features_clean"
current_stores_table = f"{catalog}.{bronze_schema}.current_stores_ne"
output_table = f"{catalog}.{silver_schema}.whitespace_locations"

print(f"Catalog: {catalog}")
print(f"H3 features: {h3_features_table}")
print(f"Current stores: {current_stores_table}")
print(f"Output table: {output_table}")
print(f"State filter: {state_filter}")
print(f"Location ID start: {location_id_start}")
print(f"Min distance to existing store: {min_distance_to_store_miles} miles")
print(f"Min candidate spacing: {min_candidate_spacing_miles} miles")
print(f"Percentile threshold: {percentile_threshold} (top {100*(1-percentile_threshold):.0f}% by POI count AND population)")

## Load H3 Features and Filter to Top Percentile (Population AND POI Count)

In [ ]:
# Load clean H3 features and filter to state_filter for whitespace locations
# Note: h3_features_clean contains all training states (MA, CT, NJ, MD)
h3_features_all = spark.table(h3_features_table)

# Parse state_filter - handle both single state and comma-separated list
state_list = [s.strip() for s in state_filter.split(",") if s.strip()]
print(f"Target states for whitespace locations: {state_list}")

# Filter to target state(s) for whitespace locations
if len(state_list) == 1:
    h3_features = h3_features_all.filter(col("state_abbr") == state_list[0])
else:
    h3_features = h3_features_all.filter(col("state_abbr").isin(state_list))

total_cells_all = h3_features_all.count()
total_cells = h3_features.count()
print(f"Total H3 cells in h3_features_clean: {total_cells_all:,}")
print(f"Filtered to {state_list}: {total_cells:,} H3 cells")

# Guard against empty data
if total_cells == 0:
    raise RuntimeError(
        f"No H3 cells found for state(s) {state_list} in {h3_features_table}. "
        f"Please ensure the silver pipeline has run and the table contains data for these states."
    )

# Filter to cells with valid data for both metrics
cells_with_data = h3_features.filter(
    (col("total_poi_count") > 0) & (col("population") > 0)
)
cells_with_data_count = cells_with_data.count()

if cells_with_data_count == 0:
    raise RuntimeError(
        f"No H3 cells with POI and population data found for state(s) {state_list}. "
        f"Please ensure H3 feature engineering has completed."
    )

print(f"H3 cells with both POI and population data: {cells_with_data_count:,}")

# Calculate percentile thresholds for BOTH total_poi_count AND population
thresholds = cells_with_data.selectExpr(
    f"percentile_approx(total_poi_count, {percentile_threshold}) as poi_threshold",
    f"percentile_approx(population, {percentile_threshold}) as pop_threshold"
).collect()[0]

poi_threshold = thresholds['poi_threshold']
pop_threshold = thresholds['pop_threshold']

print(f"\n{percentile_threshold*100:.0f}th percentile thresholds:")
print(f"  - total_poi_count: {poi_threshold}")
print(f"  - population: {pop_threshold}")

# Filter to top (1 - percentile_threshold)% by BOTH total POI count AND population
# H3 cells must meet BOTH criteria to be considered for expansion
top_h3_cells = h3_features.filter(
    (col("total_poi_count") >= poi_threshold) &
    (col("population") >= pop_threshold)
)

top_count = top_h3_cells.count()
pct_of_total = (100 * top_count / total_cells) if total_cells > 0 else 0
print(f"\nH3 cells in top {100*(1-percentile_threshold):.0f}% for BOTH metrics: {top_count:,} ({pct_of_total:.1f}%)")

## Calculate H3 Centroids

In [ ]:
# Calculate H3 cell centroids for lat/lon
h3_with_centroids = top_h3_cells.select(
    col("h3_cell_id"),
    col("state_abbr"),
    col("total_poi_count"),
    col("population"),
    col("urbanity"),
    col("urbanity_category"),
    expr("h3_centeraswkt(h3_cell_id)").alias("center_wkt")
).withColumn(
    "center_point", expr("ST_GeomFromWKT(center_wkt, 4326)")
).withColumn(
    "latitude", expr("ST_Y(center_point)")
).withColumn(
    "longitude", expr("ST_X(center_point)")
).drop("center_wkt", "center_point")

print(f"Calculated centroids for {h3_with_centroids.count():,} H3 cells")
display(h3_with_centroids.limit(5))

## Load Current Stores for Distance Calculation

In [ ]:
# Load current stores from ALL states for distance calculation
# This ensures we calculate distance to the nearest store even if it's in a neighboring state
current_stores = spark.table(current_stores_table).select(
    col("location_id").alias("store_id"),
    col("state").alias("store_state"),
    col("latitude").alias("store_lat"),
    col("longitude").alias("store_lon")
)

store_count = current_stores.count()
print(f"Loaded {store_count} current stores (all states) for distance calculation")

# Show store distribution by state
print("\nCurrent stores by state:")
display(current_stores.groupBy("store_state").count().orderBy("store_state"))

if store_count == 0:
    print(f"\n⚠️  WARNING: No current stores found")
    print(f"Distance to nearest store will be set to NULL")

## Calculate Haversine Distance to Nearest Current Store

In [ ]:
# Haversine distance calculation using Spark SQL (in MILES)
# Formula: 2 * R * arcsin(sqrt(sin²((lat2-lat1)/2) + cos(lat1)*cos(lat2)*sin²((lon2-lon1)/2)))
# R = 3959 miles (Earth's radius)

if store_count > 0:
    # Cross join candidates with stores and calculate distances
    candidates_with_distances = h3_with_centroids.crossJoin(
        broadcast(current_stores)
    ).withColumn(
        "distance_miles",
        expr("""
            2 * 3959 * asin(
                sqrt(
                    pow(sin(radians(store_lat - latitude) / 2), 2) +
                    cos(radians(latitude)) * cos(radians(store_lat)) *
                    pow(sin(radians(store_lon - longitude) / 2), 2)
                )
            )
        """)
    )
    
    # Rank stores by distance per H3 cell (keep nearest store + distance)
    from pyspark.sql.window import Window
    
    ranked = candidates_with_distances.withColumn(
        "rank", row_number().over(
            Window.partitionBy("h3_cell_id").orderBy("distance_miles")
        )
    )
    
    # Keep only the nearest store for each H3 cell
    min_distance_per_cell = ranked.filter(col("rank") == 1).select(
        "h3_cell_id", "state_abbr", "total_poi_count", "population",
        "urbanity", "urbanity_category", "latitude", "longitude",
        col("distance_miles").alias("distance_to_nearest_current_store"),
        col("store_id").alias("nearest_store_id")
    )
    
    pre_filter_count = min_distance_per_cell.count()
    print(f"Calculated distances for {pre_filter_count:,} candidate locations")
    
    # Show distance distribution before filtering
    print("\nDistance to nearest current store (miles) - BEFORE filtering:")
    display(min_distance_per_cell.select("distance_to_nearest_current_store").summary())
    
    # FILTER: Remove candidates too close to existing stores
    min_distance_per_cell = min_distance_per_cell.filter(
        col("distance_to_nearest_current_store") >= min_distance_to_store_miles
    )
    
    post_filter_count = min_distance_per_cell.count()
    removed_count = pre_filter_count - post_filter_count
    print(f"\n✓ Filtered out {removed_count:,} candidates within {min_distance_to_store_miles} miles of existing stores")
    print(f"Remaining candidates: {post_filter_count:,}")
    
else:
    # No stores - set distance and store_id to NULL
    min_distance_per_cell = h3_with_centroids.withColumn(
        "distance_to_nearest_current_store", lit(None).cast("double")
    ).withColumn(
        "nearest_store_id", lit(None).cast("string")
    )
    print("No current stores found - distance set to NULL")

## Apply Greedy Spacing to Reduce Candidate Overlap

Phase 3.2 OPTIMIZATION: Uses H3 k-ring spatial indexing instead of O(n²) pairwise haversine checks.

Select candidates ensuring minimum spacing between them to avoid oversaturation.
Candidates are processed in descending POI count order - higher-POI locations are prioritized.

For H3 resolution 8, each k-ring adds approximately 0.29 miles radius. The algorithm:
1. Sort candidates by POI count descending
2. For each candidate, check if its H3 cell is in the excluded set
3. If not excluded, select it and add all cells in k-ring to excluded set
4. This achieves O(n) complexity vs O(n²) for pairwise distance checks

In [ ]:
# Phase 3.2 OPTIMIZATION: Greedy spacing selection using H3 k-ring indexing
# Instead of O(n²) pairwise haversine checks, use H3 spatial indexing for O(n) filtering
# 
# Strategy: For each selected candidate, mark all H3 cells within k-ring distance as excluded
# H3 resolution 8 cells have ~0.46 km edge length (~0.29 miles)

import h3

def greedy_select_spaced_candidates_h3(candidates_pdf, min_spacing_miles):
    """
    Greedy selection using H3 k-ring for efficient spatial filtering.
    
    For H3 resolution 8:
    - Average edge length: ~0.46 km (~0.29 miles)
    - k=1 ring covers ~0.9 miles diameter
    - k=2 ring covers ~1.7 miles diameter
    - k=3 ring covers ~2.6 miles diameter
    - k=5 ring covers ~4.3 miles diameter
    
    Input DataFrame must be sorted by total_poi_count DESC.
    Returns list of selected h3_cell_ids.
    """
    # Calculate k-ring distance based on min_spacing_miles
    # Each H3-8 k-ring adds ~0.29 miles radius
    # Using slightly larger k to ensure coverage
    k_ring_distance = max(1, int(min_spacing_miles / 0.29) + 1)
    
    print(f"Using H3 k-ring distance: k={k_ring_distance} for {min_spacing_miles} mile spacing")
    
    selected = []
    excluded_cells = set()
    
    for _, row in candidates_pdf.iterrows():
        h3_cell = row['h3_cell_id']
        
        # Skip if this cell is already excluded
        if h3_cell in excluded_cells:
            continue
        
        # This cell is valid - select it
        selected.append(h3_cell)
        
        # Mark all cells in k-ring as excluded (including the cell itself)
        try:
            neighbors = h3.grid_disk(h3_cell, k_ring_distance)
            excluded_cells.update(neighbors)
        except Exception as e:
            # Fallback: just exclude the cell itself
            excluded_cells.add(h3_cell)
    
    return selected

# Collect to Pandas (sorted by POI count descending)
print(f"Applying H3-optimized greedy spacing filter (min {min_candidate_spacing_miles} miles between candidates)...")
pre_spacing_count = min_distance_per_cell.count()

candidates_pdf = min_distance_per_cell.orderBy(F.desc("total_poi_count")).toPandas()

# Apply H3-based greedy selection
selected_h3_ids = greedy_select_spaced_candidates_h3(candidates_pdf, min_candidate_spacing_miles)

print(f"Greedy selection: {len(selected_h3_ids):,} candidates selected from {pre_spacing_count:,}")
print(f"Removed {pre_spacing_count - len(selected_h3_ids):,} overlapping candidates")

# Filter Spark DataFrame to only selected candidates
min_distance_per_cell = min_distance_per_cell.filter(
    col("h3_cell_id").isin(selected_h3_ids)
)

final_count = min_distance_per_cell.count()
print(f"\n✓ Final candidate count after H3-optimized greedy spacing: {final_count:,}")

## Reverse Geocode to Infer City Names

Use Nominatim (OpenStreetMap) reverse geocoding to infer city/town names from lat/long coordinates. This runs on the driver with rate limiting to respect API limits.

In [ ]:
# Spatial join with Census County Subdivisions to get city/town names
# In New England, county subdivisions (towns) cover ALL land area, unlike Census Places
# Uses pygris to fetch boundaries - much faster than API geocoding

%pip install pygris --quiet

import pygris
from pygris import county_subdivisions

# Get unique states from candidates
candidate_states = [row['state_abbr'] for row in min_distance_per_cell.select("state_abbr").distinct().collect()]
print(f"Fetching County Subdivisions for states: {candidate_states}")

# Fetch County Subdivisions (towns in New England) for each state
all_subdivisions = []
for state in candidate_states:
    try:
        state_cousub = county_subdivisions(state=state, cb=True, year=2023)
        state_cousub['state_abbr'] = state
        all_subdivisions.append(state_cousub)
        print(f"  {state}: {len(state_cousub)} subdivisions")
    except Exception as e:
        print(f"  {state}: Failed to fetch subdivisions - {e}")

if all_subdivisions:
    import pandas as pd
    cousub_gdf = pd.concat(all_subdivisions, ignore_index=True)
    
    # Convert to Spark with geometry
    cousub_gdf['geometry_wkt'] = cousub_gdf['geometry'].apply(lambda g: g.wkt if g else None)
    cousub_pdf = cousub_gdf[['NAME', 'geometry_wkt']].copy()
    cousub_pdf.columns = ['place_name', 'geometry_wkt']
    
    cousub_df = spark.createDataFrame(cousub_pdf).withColumn(
        "place_geometry", expr("ST_GeomFromText(geometry_wkt, 4326)")
    ).drop("geometry_wkt")
    
    print(f"\n✓ Loaded {cousub_df.count()} County Subdivisions for spatial join")
    
    # Create point geometry for candidates with SRID 4326 to match boundaries
    candidates_with_point = min_distance_per_cell.withColumn(
        "candidate_point", expr("ST_SetSRID(ST_Point(longitude, latitude), 4326)")
    )
    
    # Spatial join: find which subdivision each candidate is in
    joined = candidates_with_point.crossJoin(
        broadcast(cousub_df)
    ).filter(
        expr("ST_Contains(place_geometry, candidate_point)")
    ).select(
        candidates_with_point.columns + ["place_name"]
    ).drop("candidate_point")
    
    # For candidates not in any subdivision, keep them with Unknown city
    candidates_in_places = joined.select("h3_cell_id", col("place_name").alias("city"))
    
    min_distance_per_cell = min_distance_per_cell.join(
        candidates_in_places,
        on="h3_cell_id",
        how="left"
    ).withColumn(
        "city", F.coalesce(col("city"), lit("Unknown"))
    ).withColumn(
        "zip_code", lit("NA")  # ZIP codes would need separate boundary data
    )
    
    # Show city distribution
    city_counts = min_distance_per_cell.groupBy("city").count().orderBy(F.desc("count"))
    matched = min_distance_per_cell.filter(col("city") != "Unknown").count()
    total = min_distance_per_cell.count()
    print(f"\n✓ Matched {matched}/{total} candidates ({100*matched/total:.1f}%) to towns/cities")
    print("\nTop cities/towns:")
    display(city_counts.limit(15))
    
else:
    print("⚠️ No subdivision data available - using placeholder values")
    min_distance_per_cell = min_distance_per_cell.withColumn(
        "city", lit("Unknown")
    ).withColumn(
        "zip_code", lit("NA")
    )

## Format Output Schema

In [ ]:
# Add row numbers for location_id
window_spec = Window.orderBy(F.desc("total_poi_count"))

whitespace_locations = min_distance_per_cell.withColumn(
    "row_num", row_number().over(window_spec)
).withColumn(
    "location_id", (lit(location_id_start) + col("row_num") - 1).cast("int")
).drop("row_num")

# Add standardized columns - use geocoded city/zip and state_abbr from H3 features data
whitespace_locations = whitespace_locations.select(
    col("location_id"),
    lit("Expansion Candidate").alias("store_type"),
    col("latitude").cast("double"),
    col("longitude").cast("double"),
    lit("NA").alias("address"),
    F.coalesce(col("city"), lit("Unknown")).alias("city"),  # Use geocoded city
    F.coalesce(col("zip_code"), lit("NA")).alias("zip_code"),  # Use geocoded zip
    col("state_abbr").alias("state"),
    lit("US").alias("country_code"),
    lit("H3_CENTROID").alias("geo_accuracy"),
    F.round(col("distance_to_nearest_current_store"), 2).alias("distance_to_nearest_current_store"),
    col("nearest_store_id"),
    col("h3_cell_id"),
    col("total_poi_count"),
    col("population"),
    col("urbanity")
)

# Add processing timestamp
whitespace_locations = whitespace_locations.withColumn(
    "processing_timestamp", F.current_timestamp()
)

print(f"Generated {whitespace_locations.count():,} whitespace locations")
print(f"\nOutput schema:")
whitespace_locations.printSchema()

# Show city distribution
print("\nCity distribution:")
display(whitespace_locations.groupBy("city").count().orderBy(F.desc("count")).limit(15))

## Write to Silver Table

In [ ]:
# Write to Delta table
(
    whitespace_locations
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print(f"\n✓ Written {whitespace_locations.count():,} whitespace locations to {output_table}")

## Validation

In [ ]:
print("=" * 80)
print("WHITESPACE LOCATIONS VALIDATION")
print("=" * 80)

# Summary statistics
print("\nSummary Statistics:")
display(spark.sql(f"""
    SELECT
        COUNT(*) as total_locations,
        COUNT(DISTINCT h3_cell_id) as unique_h3_cells,
        COUNT(DISTINCT city) as unique_cities,
        COUNT(nearest_store_id) as locations_with_nearest_store,
        MIN(location_id) as min_location_id,
        MAX(location_id) as max_location_id,
        ROUND(AVG(total_poi_count), 0) as avg_poi_count,
        ROUND(AVG(population), 0) as avg_population,
        ROUND(AVG(distance_to_nearest_current_store), 2) as avg_distance_to_store_miles,
        ROUND(MIN(distance_to_nearest_current_store), 2) as min_distance_miles,
        ROUND(MAX(distance_to_nearest_current_store), 2) as max_distance_miles
    FROM {output_table}
"""))

# Distribution by urbanity
print("\nBy Urbanity:")
display(spark.sql(f"""
    SELECT
        urbanity,
        COUNT(*) as location_count,
        ROUND(AVG(total_poi_count), 0) as avg_poi_count,
        ROUND(AVG(population), 0) as avg_population,
        ROUND(AVG(distance_to_nearest_current_store), 2) as avg_distance_miles
    FROM {output_table}
    GROUP BY urbanity
    ORDER BY location_count DESC
"""))

# Distribution by city
print("\nTop Cities:")
display(spark.sql(f"""
    SELECT
        city,
        COUNT(*) as location_count,
        ROUND(AVG(total_poi_count), 0) as avg_poi_count
    FROM {output_table}
    GROUP BY city
    ORDER BY location_count DESC
    LIMIT 15
"""))

# Sample records
print("\nSample whitespace locations:")
display(spark.table(output_table).select(
    "location_id", "store_type", "city", "state", "zip_code",
    "latitude", "longitude", "distance_to_nearest_current_store",
    "total_poi_count", "population", "urbanity"
).orderBy("location_id").limit(10))

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)